## Structure Hierarchical Classifier

In [1]:
import torch
import torch.nn as nn

class HierarchicalClassifier(nn.Module):
    def __init__(self, binary_clf: nn.Module, multi_clf: nn.Module, device="cpu"):
        """
        Args:
            binary_clf (nn.Module): Binary classifier (N vs F).
            multi_clf (nn.Module): Multi-class classifier (B, I, O).
            device (str): Device to run the model.
        """
        super().__init__()
        self.binary_clf = binary_clf.to(device)
        self.multi_clf = multi_clf.to(device)
        self.device = device

        # Map final labels: 0=N, 1=B, 2=I, 3=O
        self.final_classes = {0: "N", 1: "B", 2: "I", 3: "O"}

    def forward(self, x):
        """
        Forward pass through the hierarchical classifier.
        
        Args:
            x (torch.Tensor): Input batch [batch_size, features].

        Returns:
            torch.Tensor: Final predictions (0=N, 1=B, 2=I, 3=O).
        """
        x = x.to(self.device)

        # Step 1: Binary classification (N=0, F=1)
        bin_out = self.binary_clf(x)  
        bin_pred = torch.argmax(bin_out, dim=1)

        final_pred = []
        for i, pred in enumerate(bin_pred):
            if pred.item() == 0:  
                # Class N (normal bearing)
                final_pred.append(0)
            else:
                # Step 2: Multi-class classification (B=1, I=2, O=3)
                multi_out = self.multi_clf(x[i].unsqueeze(0))
                multi_pred = torch.argmax(multi_out, dim=1).item()
                final_pred.append(multi_pred + 1)  # shift to {1,2,3}
        
        return torch.tensor(final_pred, device=self.device)

    def predict(self, x):
        """Convenience method for prediction with labels."""
        preds = self.forward(x)
        return [self.final_classes[p.item()] for p in preds]


In [ ]:
# Suponha que você já tenha treinado:
# binary_clf -> classificador N vs F
# multi_clf  -> classificador B/I/O

hier_clf = HierarchicalClassifier(binary_clf, multi_clf, device="cuda")

# Exemplo de inferência
x = torch.randn(8, 250).to("cuda")  # batch de 8 amostras (250 pontos cada)
preds = hier_clf.predict(x)
print(preds)  # -> ['N', 'B', 'I', 'O', ...]


## Dataset

In [2]:
import os
from glob import glob
import numpy as np
import torch
from torch.utils.data import Dataset

# Label maps
LABELS = {
    "bin": {"N": 0, "F": 1},        # non-fault vs fault
    "bio": {"B": 0, "I": 1, "O": 2} # ball, inner, outer
}

class BearingNPYDataset(Dataset):
    """
    Loads .npy 1D signals organized as:
      root/bin/(N_1_0.npy,...,F_2_1.npy)   -> task="bin"
      root/bio/(B_5_2.npy,...,I_9_5.npy,...) -> task="bio"
    Label is inferred from the first char in filename.
    """

    def __init__(self, root_dir, task="bin", transform=None, target_transform=None, return_path=False):
        self.root_dir = root_dir
        self.task = task.lower().strip()
        if self.task not in LABELS:
            raise ValueError("task must be 'bin' or 'bio'")
        self.map = LABELS[self.task]
        self.transform = transform
        self.target_transform = target_transform
        self.return_path = return_path

        # collect files
        paths = sorted(glob(os.path.join(root_dir, "*.npy")))
        if not paths:
            raise FileNotFoundError("No .npy files found in %s" % root_dir)

        # keep only files whose first char is in label map
        self.samples = [p for p in paths if os.path.basename(p)[0] in self.map]
        if not self.samples:
            raise FileNotFoundError("No valid labeled files (by prefix) in %s" % root_dir)

        # store class names in index order
        inv = sorted(self.map.items(), key=lambda x: x[1])
        self.classes = [k for k, _ in inv]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        x = np.load(path)                 # expects 1D numpy array
        x = torch.from_numpy(x).float()   # to float32 tensor
        if x.ndim == 1:
            x = x.unsqueeze(0)            # shape [1, L] for Conv1d

        y_char = os.path.basename(path)[0]
        y = torch.tensor(self.map[y_char], dtype=torch.long)

        if self.transform is not None:
            x = self.transform(x)
        if self.target_transform is not None:
            y = self.target_transform(y)

        if self.return_path:
            return x, y, path
        return x, y
